#Kaggle to S3
> Send all the historic data from kaggle to s3

In [0]:
%pip install kaggle boto3

In [0]:
%run /Repos/nikum.vedansh@gmail.com/formula-one-project/configs/credentials

In [0]:
import boto3
import json
import os
from datetime import datetime
import kaggle
import pandas as pd
from io import StringIO

In [0]:
s3 = boto3.client(
    's3',
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=AWS_REGION
)

WATERMARK_KEY = "watermark/last_updated.json"

def get_watermark():
    try:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=WATERMARK_KEY)
        data = json.loads(obj['Body'].read())
        return datetime.strptime(data["last_updated"], "%Y-%m-%d")
    except:
        # No watermark yet — this is the first full load
        return None

def set_watermark(date_str):
    s3.put_object(
        Bucket=S3_BUCKET,
        Key=WATERMARK_KEY,
        Body=json.dumps({"last_updated": date_str})
    )
    print(f"Watermark updated to {date_str}")

watermark = get_watermark()
print(f"Current watermark: {watermark if watermark else 'None — full load'}")

In [0]:
kaggle.api.authenticate()

datasets = {
    'jtrotman/formula-1-race-data': 'race_data',
    'jtrotman/formula-1-race-events': 'race_events'
}

for dataset_id, folder in datasets.items():
    print(f"\nDownloading {dataset_id}...")
    kaggle.api.dataset_download_files(dataset_id, path=f'/tmp/{folder}', unzip=True)
    print(f"Downloaded to /tmp/{folder}")

In [0]:
# Load races.csv to use as date reference
races_df = pd.read_csv('/tmp/race_data/races.csv')
races_df['date'] = pd.to_datetime(races_df['date'], errors='coerce')

# Get latest race date for watermark update
latest_date = races_df['date'].max().strftime("%Y-%m-%d")

uploaded = []
skipped = []

for folder in ['race_data', 'race_events']:
    for file in os.listdir(f'/tmp/{folder}'):
        if not file.endswith('.csv'):
            continue

        df = pd.read_csv(f'/tmp/{folder}/{file}')

        if watermark is None:
            # First full load — upload everything to full_load/
            key = f'{folder}/full_load/{file}'
            csv_buffer = StringIO()
            df.to_csv(csv_buffer, index=False)
            s3.put_object(Bucket=S3_BUCKET, Key=key, Body=csv_buffer.getvalue())
            uploaded.append(file)
            print(f"  ✅ Full load: {file}")

        else:
            # Incremental load — filter to new rows only
            if 'raceId' in df.columns:
                df = df.merge(races_df[['raceId', 'date']], on='raceId', how='left')
                new_rows = df[df['date'] > watermark]
            elif 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'], errors='coerce')
                new_rows = df[df['date'] > watermark]
            else:
                # No date column — skip incremental, already have full load
                skipped.append(file)
                continue

            if new_rows.empty:
                skipped.append(file)
                continue

            # Drop the merged date column if added
            if 'date_y' in new_rows.columns:
                new_rows = new_rows.drop(columns=['date_y'])
            if 'date_x' in new_rows.columns:
                new_rows = new_rows.rename(columns={'date_x': 'date'})

            timestamp = datetime.now().strftime("%Y%m%d")
            key = f'{folder}/incremental/{file.replace(".csv", "")}_{timestamp}.csv'
            csv_buffer = StringIO()
            new_rows.to_csv(csv_buffer, index=False)
            s3.put_object(Bucket=S3_BUCKET, Key=key, Body=csv_buffer.getvalue())
            uploaded.append(file)
            print(f"  ✅ Incremental: {file} — {len(new_rows)} new rows")

# Update watermark
set_watermark(latest_date)

print(f"\n✅ Uploaded: {len(uploaded)}")
print(f"⏩ Skipped: {len(skipped)}")